# 🔗 Notebook 3: Advanced Transformations, Window Functions & Distributed Joins
### *From Multi-Dimensional Aggregations to Complex Windows and Broadcast Join Mechanics*

> **Companion Video Reference:** [YouTube: PySpark Tutorial | Full Course](https://www.youtube.com/watch?v=94w6hPk7nkM) by Ansh Lamba & [Advancing Analytics Joins Deep-Dive](https://www.youtube.com/@AdvancingAnalytics)

---

## 🎯 What Will You Learn in This Notebook?

In enterprise data pipelines and machine learning feature stores, simple filters and column selects are only the starting point. The real heavy lifting comes from:
1. **Multi-Metric Aggregations & Pivoting:** Grouping and summarizing across multiple dimensions simultaneously.
2. **Window Functions:** Computing running totals, moving averages, row rankings, and lead/lag time differences without collapsing rows.
3. **Distributed Joins Under the Hood:** Understanding why some joins crash your cluster and how to optimize them.
4. **Sort-Merge Join (SMJ) vs. Broadcast Hash Join (BHJ):** Eliminating network shuffle bottlenecks using `broadcast()`.

---


## 🪜 1. Window Functions: The Secret Weapon of Senior Engineers

Unlike `groupBy()`, which reduces multiple rows into a single summary row, **Window Functions** compute an aggregate over a sliding "window" of rows while **retaining the full granularity of every individual row**!

```
                    WINDOW SPECIFICATION (Window.partitionBy("dept").orderBy("salary"))
┌───────────┬──────────────┬────────┬─────────────────────────────────────────────────┐
│ Employee  │ Department   │ Salary │ Window Result (e.g., Rank or Running Total)     │
├───────────┼──────────────┼────────┼─────────────────────────────────────────────────┤
│ Alice     │ Engineering  │ 120k   │ Rank 1  | Running Total: 120k                   │
│ Bob       │ Engineering  │ 110k   │ Rank 2  | Running Total: 230k                   │
│ Charlie   │ Engineering  │ 90k    │ Rank 3  | Running Total: 320k                   │
├───────────┼──────────────┼────────┼─────────────────────────────────────────────────┤
│ David     │ Sales        │ 95k    │ Rank 1  | Running Total: 95k                    │
│ Emma      │ Sales        │ 80k    │ Rank 2  | Running Total: 175k                   │
└───────────┴──────────────┴────────┴─────────────────────────────────────────────────┘
```

### The 3 Types of Window Operations:
1. **Ranking Functions:** `row_number()`, `rank()`, `dense_rank()`, `percent_rank()`
2. **Analytic Functions:** `lead(col, offset)` (look ahead), `lag(col, offset)` (look behind)
3. **Aggregate Functions:** `sum()`, `avg()`, `min()`, `max()` with frame bounds (`rowsBetween`)


## ⚡ 2. Distributed Join Strategies: How Spark Joins Data

When you write `df_a.join(df_b, "id")`, what actually happens across the cluster?

### Strategy A: Shuffle Sort-Merge Join (SMJ) - Default for Large Tables
```
Table A (Big) ──▶ [Hash on Key] ──▶ [Network SHUFFLE] ──▶ [Sort Keys on Disk/RAM] ──┐
                                                                                     ├──▶ [Merge Stream]
Table B (Big) ──▶ [Hash on Key] ──▶ [Network SHUFFLE] ──▶ [Sort Keys on Disk/RAM] ──┘
```
- Both tables are hashed and transmitted across the network so rows with the same key end up on the same executor.
- Then both sides are sorted by the join key and merged.
- **Cost:** Extremely expensive network I/O and disk spill if memory is tight!

### Strategy B: Broadcast Hash Join (BHJ) - The 100x Speedup for Dimension Lookups!
```
Driver reads Small Table (e.g. 50 MB)
Driver transmits a complete copy to EVERY Executor RAM!
Executor 1: [ Big Table Partition 1 ] ──▶ In-Memory Hash Lookup (Zero Network Shuffle!)
Executor 2: [ Big Table Partition 2 ] ──▶ In-Memory Hash Lookup (Zero Network Shuffle!)
Executor 3: [ Big Table Partition 3 ] ──▶ In-Memory Hash Lookup (Zero Network Shuffle!)
```
- **Zero Shuffle!** Big table never moves over the network.
- Spark automatically applies BHJ if the small table is $< 10\text{ MB}$ (`spark.sql.autoBroadcastJoinThreshold`), or you can force it using `broadcast(small_df)`!


## 🚀 Hands-On Lab: Session Setup


In [1]:
import os
import sys
from pathlib import Path

# Configure environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    avg as spark_avg,
    count,
    max as spark_max,
    min as spark_min,
    row_number,
    rank,
    dense_rank,
    lead,
    lag,
    broadcast,
    round as spark_round
)

# Initialize local SparkSession
spark = SparkSession.builder \
    .appName("PySpark_Advanced_Transformations_and_Joins") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession initialized successfully!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 11:40:57 WARN Utils: Your hostname, Prafull-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.161 instead (on interface en0)
26/09/22 11:40:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/22 11:40:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession initialized successfully!


### Creating Rich Datasets: Sales Transactions & Regional Lookups


In [2]:
# 1. Detailed Sales Transactions
sales_data = [
    (1, "Store_A", "Electronics", "Laptop", 1200.0, "2024-01-01"),
    (2, "Store_A", "Electronics", "Phone", 800.0, "2024-01-02"),
    (3, "Store_A", "Furniture", "Desk", 350.0, "2024-01-03"),
    (4, "Store_A", "Electronics", "Tablet", 500.0, "2024-01-04"),
    (5, "Store_B", "Electronics", "Laptop", 1250.0, "2024-01-01"),
    (6, "Store_B", "Furniture", "Chair", 150.0, "2024-01-02"),
    (7, "Store_B", "Furniture", "Desk", 300.0, "2024-01-03"),
    (8, "Store_B", "Electronics", "Monitor", 280.0, "2024-01-04"),
    (9, "Store_C", "Electronics", "Phone", 850.0, "2024-01-01"),
    (10, "Store_C", "Furniture", "Chair", 180.0, "2024-01-02"),
]

sales_cols = ["sale_id", "store_id", "department", "product", "amount", "sale_date"]
sales_df = spark.createDataFrame(sales_data, sales_cols)

# 2. Store Dimension (Metadata lookup table - perfect for Broadcast Join!)
store_data = [
    ("Store_A", "New York", "Northeast", 45000),
    ("Store_B", "Chicago", "Midwest", 38000),
    ("Store_C", "Austin", "South", 52000),
    ("Store_D", "Seattle", "Northwest", 41000), # Store with no sales yet
]
store_cols = ["store_id", "city", "region", "sqft"]
store_df = spark.createDataFrame(store_data, store_cols)

print("✅ Sales DataFrame:")
sales_df.show(5)
print("✅ Store Dimension DataFrame:")
store_df.show()


✅ Sales DataFrame:


+-------+--------+-----------+-------+------+----------+
|sale_id|store_id| department|product|amount| sale_date|
+-------+--------+-----------+-------+------+----------+
|      1| Store_A|Electronics| Laptop|1200.0|2024-01-01|
|      2| Store_A|Electronics|  Phone| 800.0|2024-01-02|
|      3| Store_A|  Furniture|   Desk| 350.0|2024-01-03|
|      4| Store_A|Electronics| Tablet| 500.0|2024-01-04|
|      5| Store_B|Electronics| Laptop|1250.0|2024-01-01|
+-------+--------+-----------+-------+------+----------+
only showing top 5 rows
✅ Store Dimension DataFrame:
+--------+--------+---------+-----+
|store_id|    city|   region| sqft|
+--------+--------+---------+-----+
| Store_A|New York|Northeast|45000|
| Store_B| Chicago|  Midwest|38000|
| Store_C|  Austin|    South|52000|
| Store_D| Seattle|Northwest|41000|
+--------+--------+---------+-----+



## 📊 3. Multi-Metric Aggregations & Pivot Tables

Let's compute multiple aggregates simultaneously and pivot departments into distinct columns:


In [3]:
# 1. Multi-metric summary per Store
summary_df = sales_df.groupBy("store_id") \
    .agg(
        count("sale_id").alias("total_transactions"),
        spark_sum("amount").alias("total_sales"),
        spark_round(spark_avg("amount"), 2).alias("avg_transaction_value"),
        spark_max("amount").alias("highest_single_sale")
    ).orderBy("total_sales", ascending=False)

print("✅ Store Sales Performance Summary:")
summary_df.show()

# 2. Pivot Table: Total Revenue by Store (rows) across Departments (columns)
pivot_df = sales_df.groupBy("store_id") \
    .pivot("department", ["Electronics", "Furniture"]) \
    .agg(spark_sum("amount")) \
    .na.fill(0.0)

print("✅ Department Revenue Pivoted by Store:")
pivot_df.show()


✅ Store Sales Performance Summary:


+--------+------------------+-----------+---------------------+-------------------+
|store_id|total_transactions|total_sales|avg_transaction_value|highest_single_sale|
+--------+------------------+-----------+---------------------+-------------------+
| Store_A|                 4|     2850.0|                712.5|             1200.0|
| Store_B|                 4|     1980.0|                495.0|             1250.0|
| Store_C|                 2|     1030.0|                515.0|              850.0|
+--------+------------------+-----------+---------------------+-------------------+

✅ Department Revenue Pivoted by Store:


+--------+-----------+---------+
|store_id|Electronics|Furniture|
+--------+-----------+---------+
| Store_B|     1530.0|    450.0|
| Store_C|      850.0|    180.0|
| Store_A|     2500.0|    350.0|
+--------+-----------+---------+



## 🪟 4. Window Functions in Action

### A. Ranking Functions (`row_number`, `rank`, `dense_rank`)
Let's find the top-selling product in each store:


In [4]:
# Window partition by Store, ordered by sale amount descending
store_window = Window.partitionBy("store_id").orderBy(col("amount").desc())

ranked_sales_df = sales_df.select(
    col("store_id"),
    col("department"),
    col("product"),
    col("amount"),
    row_number().over(store_window).alias("row_num"),
    rank().over(store_window).alias("rank"),
    dense_rank().over(store_window).alias("dense_rank")
)

print("✅ Ranked Sales within each store:")
ranked_sales_df.show(truncate=False)

# Filtering Top 1 sale per store
top_sale_per_store = ranked_sales_df.filter(col("row_num") == 1)
print("🏆 Top 1 sale for each store:")
top_sale_per_store.show()


✅ Ranked Sales within each store:


+--------+-----------+-------+------+-------+----+----------+
|store_id|department |product|amount|row_num|rank|dense_rank|
+--------+-----------+-------+------+-------+----+----------+
|Store_A |Electronics|Laptop |1200.0|1      |1   |1         |
|Store_A |Electronics|Phone  |800.0 |2      |2   |2         |
|Store_A |Electronics|Tablet |500.0 |3      |3   |3         |
|Store_A |Furniture  |Desk   |350.0 |4      |4   |4         |
|Store_B |Electronics|Laptop |1250.0|1      |1   |1         |
|Store_B |Furniture  |Desk   |300.0 |2      |2   |2         |
|Store_B |Electronics|Monitor|280.0 |3      |3   |3         |
|Store_B |Furniture  |Chair  |150.0 |4      |4   |4         |
|Store_C |Electronics|Phone  |850.0 |1      |1   |1         |
|Store_C |Furniture  |Chair  |180.0 |2      |2   |2         |
+--------+-----------+-------+------+-------+----+----------+

🏆 Top 1 sale for each store:


+--------+-----------+-------+------+-------+----+----------+
|store_id| department|product|amount|row_num|rank|dense_rank|
+--------+-----------+-------+------+-------+----+----------+
| Store_A|Electronics| Laptop|1200.0|      1|   1|         1|
| Store_B|Electronics| Laptop|1250.0|      1|   1|         1|
| Store_C|Electronics|  Phone| 850.0|      1|   1|         1|
+--------+-----------+-------+------+-------+----+----------+



### B. Running Totals & Cumulative Aggregations
Compute a cumulative running total of sales date-by-date for each store using frame bounds:


In [5]:
# Cumulative window from earliest record to current row
cumulative_window = Window.partitionBy("store_id") \
    .orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df = sales_df.select(
    col("store_id"),
    col("sale_date"),
    col("product"),
    col("amount"),
    spark_sum("amount").over(cumulative_window).alias("cumulative_sales")
).orderBy("store_id", "sale_date")

print("✅ Cumulative Daily Sales per Store:")
running_total_df.show(truncate=False)


✅ Cumulative Daily Sales per Store:


+--------+----------+-------+------+----------------+
|store_id|sale_date |product|amount|cumulative_sales|
+--------+----------+-------+------+----------------+
|Store_A |2024-01-01|Laptop |1200.0|1200.0          |
|Store_A |2024-01-02|Phone  |800.0 |2000.0          |
|Store_A |2024-01-03|Desk   |350.0 |2350.0          |
|Store_A |2024-01-04|Tablet |500.0 |2850.0          |
|Store_B |2024-01-01|Laptop |1250.0|1250.0          |
|Store_B |2024-01-02|Chair  |150.0 |1400.0          |
|Store_B |2024-01-03|Desk   |300.0 |1700.0          |
|Store_B |2024-01-04|Monitor|280.0 |1980.0          |
|Store_C |2024-01-01|Phone  |850.0 |850.0           |
|Store_C |2024-01-02|Chair  |180.0 |1030.0          |
+--------+----------+-------+------+----------------+



### C. Lead & Lag (Time Differences)
Compare each transaction to the previous transaction in the same store:


In [6]:
# Lag window: looks at previous row
lag_window = Window.partitionBy("store_id").orderBy("sale_date")

lag_df = sales_df.select(
    col("store_id"),
    col("sale_date"),
    col("amount"),
    lag("amount", 1).over(lag_window).alias("previous_sale_amount")
).withColumn(
    "sale_diff",
    spark_round(col("amount") - col("previous_sale_amount"), 2)
)

print("✅ Transaction Difference vs Previous Sale (Lag):")
lag_df.show()


✅ Transaction Difference vs Previous Sale (Lag):


+--------+----------+------+--------------------+---------+
|store_id| sale_date|amount|previous_sale_amount|sale_diff|
+--------+----------+------+--------------------+---------+
| Store_A|2024-01-01|1200.0|                NULL|     NULL|
| Store_A|2024-01-02| 800.0|              1200.0|   -400.0|
| Store_A|2024-01-03| 350.0|               800.0|   -450.0|
| Store_A|2024-01-04| 500.0|               350.0|    150.0|
| Store_B|2024-01-01|1250.0|                NULL|     NULL|
| Store_B|2024-01-02| 150.0|              1250.0|  -1100.0|
| Store_B|2024-01-03| 300.0|               150.0|    150.0|
| Store_B|2024-01-04| 280.0|               300.0|    -20.0|
| Store_C|2024-01-01| 850.0|                NULL|     NULL|
| Store_C|2024-01-02| 180.0|               850.0|   -670.0|
+--------+----------+------+--------------------+---------+



## 🤝 5. Distributed Joins: Inner, Left, Left Semi, Left Anti & Broadcast

Let's test all join types and inspect the physical execution plan:


In [7]:
# 1. Inner Join: Only stores with active sales
inner_joined = sales_df.join(store_df, "store_id", "inner")
print(f"✅ Inner Join Count: {inner_joined.count()}")

# 2. Left Outer Join: Retains all sales, enriches with store metadata
left_joined = sales_df.join(store_df, "store_id", "left")

# 3. Left Anti Join: Finds Stores in store_df that have ZERO sales!
# (Notice: Store_D has no sales records)
anti_joined = store_df.join(sales_df, "store_id", "left_anti")
print("✅ Stores with ZERO sales (Left Anti Join):")
anti_joined.show()


✅ Inner Join Count: 10
✅ Stores with ZERO sales (Left Anti Join):
+--------+-------+---------+-----+
|store_id|   city|   region| sqft|
+--------+-------+---------+-----+
| Store_D|Seattle|Northwest|41000|
+--------+-------+---------+-----+



### Broadcast Hash Join (BHJ) vs Sort-Merge Join (SMJ)
Let's explicitly force a **Broadcast Join** using `broadcast(store_df)` and verify with `.explain()`:


In [8]:
# Force Broadcast Hash Join
broadcast_joined_df = sales_df.join(broadcast(store_df), "store_id", "left")

print("🔍 --- BROADCAST HASH JOIN PHYSICAL PLAN ---")
# Notice BroadcastHashJoin and BroadcastExchange in the output!
broadcast_joined_df.explain()

print("\n✅ Sample joined result:")
broadcast_joined_df.select("sale_id", "store_id", "city", "region", "product", "amount").show(5)


🔍 --- BROADCAST HASH JOIN PHYSICAL PLAN ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [store_id#1, sale_id#0L, department#2, product#3, amount#4, sale_date#5, city#7, region#8, sqft#9L]
   +- BroadcastHashJoin [store_id#1], [store_id#6], LeftOuter, BuildRight, false, false
      :- Scan ExistingRDD[sale_id#0L,store_id#1,department#2,product#3,amount#4,sale_date#5]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=812]
         +- Filter isnotnull(store_id#6)
            +- Scan ExistingRDD[store_id#6,city#7,region#8,sqft#9L]



✅ Sample joined result:


+-------+--------+--------+---------+-------+------+
|sale_id|store_id|    city|   region|product|amount|
+-------+--------+--------+---------+-------+------+
|      1| Store_A|New York|Northeast| Laptop|1200.0|
|      2| Store_A|New York|Northeast|  Phone| 800.0|
|      3| Store_A|New York|Northeast|   Desk| 350.0|
|      4| Store_A|New York|Northeast| Tablet| 500.0|
|      5| Store_B| Chicago|  Midwest| Laptop|1250.0|
+-------+--------+--------+---------+-------+------+
only showing top 5 rows


### Clean Session Shutdown


In [9]:
# Stop local SparkSession
spark.stop()
print("✅ SparkSession cleanly terminated.")


✅ SparkSession cleanly terminated.


## 📖 Key Takeaways: Joins & Windows Cheat Sheet

| Technique | When to Use | Performance Impact |
| :--- | :--- | :--- |
| **`row_number()`** | Deduplication, finding top-N records per category. | Fast, runs within partition boundaries. |
| **`lag()` / `lead()`** | Time-series changes, churn calculation, session delta. | Eliminates complex self-joins. |
| **`rowsBetween`** | Rolling averages (e.g. 7-day moving avg), cumulative sums. | Powerful in-memory computation. |
| **`broadcast(df)`** | Joining large fact table with small dimension table (< 100MB). | **Huge Win:** Eliminates expensive network shuffle! |
| **Sort-Merge Join** | Joining two large tables (both > 1GB). | Standard distributed join, requires network shuffle on join keys. |
| **`left_anti`** | Finding missing keys / orphan records (e.g. users with no orders). | More efficient than `WHERE col IS NULL`. |

---
**Next Step:** Move to **`04_performance_tuning_and_production_patterns.ipynb`** to master partitioning, skew salting, and production ETL!
